
# MTech Bug-Fix Agent: Colab Feasibility Check

You do not need to write code. Follow one section at a time and check its output before moving on. Every fresh runtime must start with the resource check; it chooses a model before downloading model weights.

## Run these sections in order

1. Connect to the Colab runtime and select a GPU runtime if available.
2. Run the first code cell (resource check). It prints RAM, free disk, GPU details, and recommends a model. Stop if it reports an error or insufficient resources.
3. Run **Download and install the project**.
4. Run **Confirm the command is ready**.
5. Run **Verify the three Goldset pilot records**. Continue only when each reports one pass, zero failures, and zero skips.
6. Under **Model choice for this runtime and install model libraries**, run the model-libraries installation code cell.
7. Run **Load the selected model and make one short request**. Continue only if it prints the model revision and a non-empty response.
8. Run **Refinement round: run three Goldset tasks with and without retrieval**. This makes six model attempts: both conditions for each task.
9. If any task-condition fails or is skipped, run **If a pilot task fails: inspect the saved run details** and share its output.

Skip **Check one SWE-bench Lite task and Colab capacity**; that feasibility check is complete. If the runtime restarts, begin again at step 1.

In [9]:
import json
import platform
import shutil
import sys
import torch

def gib(n):
    return round(n / (1024 ** 3), 2)

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
disk_free_gib = gib(shutil.disk_usage('/content').free)
print('Free disk at /content (GiB):', disk_free_gib)
try:
    import psutil
    ram = psutil.virtual_memory()
    ram_total_gib, ram_available_gib = gib(ram.total), gib(ram.available)
    print('System RAM total / available (GiB):', ram_total_gib, '/', ram_available_gib)
except ImportError:
    with open('/proc/meminfo', encoding='utf-8') as memfile:
        meminfo = {line.split(':', 1)[0]: int(line.split()[1]) * 1024 for line in memfile if ':' in line}
    ram_total_gib = gib(meminfo['MemTotal'])
    ram_available_gib = gib(meminfo.get('MemAvailable', meminfo['MemFree']))
    print('System RAM total / available (GiB), from /proc/meminfo:', ram_total_gib, '/', ram_available_gib)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. Select a Colab GPU runtime, rerun this resource check, and only then load a model.')
props = torch.cuda.get_device_properties(0)
gpu_total_gib = gib(props.total_memory)
gpu_free_gib = gib(torch.cuda.mem_get_info(0)[0])
print('GPU:', props.name)
print('GPU memory total / currently free (GiB):', gpu_total_gib, '/', gpu_free_gib)
print('Python executable:', sys.executable)

# Thresholds preserve download/cache and inference headroom; see PROJECT_PLAN.md.
if gpu_free_gib >= 11 and disk_free_gib >= 20:
    MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
    PILOT_MAX_INPUT_TOKENS, PILOT_MAX_NEW_TOKENS = 3072, 768
elif gpu_free_gib >= 6 and disk_free_gib >= 12:
    MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'
    PILOT_MAX_INPUT_TOKENS, PILOT_MAX_NEW_TOKENS = 2048, 512
elif gpu_free_gib >= 4.5 and disk_free_gib >= 8:
    MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
    PILOT_MAX_INPUT_TOKENS, PILOT_MAX_NEW_TOKENS = 1536, 384
else:
    raise RuntimeError('Available GPU memory or disk is below the pilot minimum. Select a larger Colab GPU/runtime before downloading model weights.')
RESOURCE_REPORT = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'disk_free_gib': disk_free_gib, 'ram_total_gib': ram_total_gib,
    'ram_available_gib': ram_available_gib, 'gpu': props.name,
    'gpu_total_gib': gpu_total_gib, 'gpu_free_gib': gpu_free_gib,
    'model_id': MODEL_ID, 'max_input_tokens': PILOT_MAX_INPUT_TOKENS,
    'max_new_tokens': PILOT_MAX_NEW_TOKENS,
}
print('Resource-based model recommendation:', MODEL_ID)
print('Pilot input/output token limits:', PILOT_MAX_INPUT_TOKENS, '/', PILOT_MAX_NEW_TOKENS)
print('Resource report JSON:', json.dumps(RESOURCE_REPORT, indent=2))

Python: 3.13.16
PyTorch: 2.11.0+cu130
Free disk at /content (GiB): 97.91
System RAM total / available (GiB): 12.67 / 8.44
GPU: Tesla T4
GPU memory total / currently free (GiB): 14.56 / 7.46
Python executable: /usr/bin/python3
Resource-based model recommendation: Qwen/Qwen2.5-Coder-3B-Instruct
Pilot input/output token limits: 2048 / 512
Resource report JSON: {
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "disk_free_gib": 97.91,
  "ram_total_gib": 12.67,
  "ram_available_gib": 8.44,
  "gpu": "Tesla T4",
  "gpu_total_gib": 14.56,
  "gpu_free_gib": 7.46,
  "model_id": "Qwen/Qwen2.5-Coder-3B-Instruct",
  "max_input_tokens": 2048,
  "max_new_tokens": 512
}


## Download and install the project

The repository is public. This cell clones it into Colab's temporary storage and installs the lightweight project package. Colab storage resets when its runtime is discarded, so rerun this notebook after a reset.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/pisalaarti05/mtech-bug-fix-agent.git'
PROJECT_DIR = Path('/content/mtech-bug-fix-agent')

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--editable', str(PROJECT_DIR)], check=True)
# Reload updated prompt/retrieval logic while keeping the already-loaded model in memory.
for module_name in ('bugfix_agent.benchmark', 'bugfix_agent.retrieval'):
    sys.modules.pop(module_name, None)
print('Project installed from:', PROJECT_DIR)
print('Refreshed benchmark/retrieval code; the loaded model remains available.')

## Confirm the command is ready

This displays the program's options. A successful result means Colab can import the project. Do not choose a model yet; first share the runtime details above.

In [11]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'bugfix_agent.cli', '--help'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'bugfix_agent.cli', '--help'], returncode=0)

## Check one SWE-bench Lite task and Colab capacity

SWE-bench Lite is a suitable issue-driven candidate, but its official evaluation runs Docker containers and needs at least 120 GB of free disk. This cell only reads one task record and checks your current Colab resources; it does not reveal the reference patch or run benchmark tests. If Docker or disk is insufficient, do not claim official SWE-bench scores from this runtime.

In [ ]:
import shutil
import subprocess
import sys

# Read one benchmark example without downloading the entire dataset.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=3.0'], check=True)
from datasets import load_dataset

tasks = load_dataset('princeton-nlp/SWE-bench_Lite', split='test', streaming=True)
task = next(iter(tasks))
print('Instance:', task['instance_id'])
print('Repository:', task['repo'])
print('Base commit:', task['base_commit'])
print('Issue description (first 1800 chars):')
print(task['problem_statement'][:1800])

disk = shutil.disk_usage('/content')
print(f'Free Colab disk: {disk.free / (1024**3):.1f} GiB')
docker = subprocess.run(['bash', '-lc', 'docker --version'], capture_output=True, text=True)
print('Docker available:', docker.returncode == 0)
if docker.returncode == 0:
    print(docker.stdout.strip())
else:
    print('Docker is not available in this runtime.')
print('Official SWE-bench harness minimum free disk: 120 GB.')


Instance: astropy__astropy-12907
Repository: astropy/astropy
Base commit: d16bfe05a744909de4b27f5875fe0d4ed41ce607
Issue description (first 1800 chars):
Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.

If however,


## Verify the three Goldset pilot records

This cell checks three compact function-level tasks from different repositories. It runs the recorded regression test against each buggy version and its reference fix. Continue to model generation only if every task reports `passed 1`, `failed 0`, and `skipped 0`. If a task cannot verify, stop and share its output; we will choose a suitable alternate task.

In [12]:

from pathlib import Path
import json
import re
import subprocess
import sys

GOLDSET_DIR = Path('/content/goldset')
GOLDSET_REVISION = 'fe4f9bfbcb46637481242af2315c6b59951710b4'
if not GOLDSET_DIR.exists():
    subprocess.run(['git', 'clone', '--quiet', '--no-checkout', 'https://github.com/andysalvo/goldset.git', str(GOLDSET_DIR)], check=True)
subprocess.run(['git', 'fetch', '--quiet', '--depth', '1', 'origin', GOLDSET_REVISION], cwd=GOLDSET_DIR, check=True)
subprocess.run(['git', 'checkout', '--quiet', 'FETCH_HEAD'], cwd=GOLDSET_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'goldset'], capture_output=True, text=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(GOLDSET_DIR), 'uv'], check=True)

for module_name in list(sys.modules):
    if module_name == 'goldset' or module_name.startswith('goldset.'):
        del sys.modules[module_name]
import importlib
if str(GOLDSET_DIR) in sys.path:
    sys.path.remove(str(GOLDSET_DIR))
sys.path.insert(0, str(GOLDSET_DIR))
importlib.invalidate_caches()
import goldset.exec
if not Path(goldset.exec.__file__).resolve().is_relative_to(GOLDSET_DIR.resolve()):
    raise RuntimeError('Goldset runner did not load from the pinned checkout.')
print('Goldset runner module:', goldset.exec.__file__)

records_by_fingerprint = {
    item['fingerprint']: item
    for item in (json.loads(line) for line in (GOLDSET_DIR / 'oracles.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())
}
PILOT_FINGERPRINTS = [
    '1f881a956841fe4f',  # ilevkivskyi/com2ann: type-comment parsing
    'd2509b7ca26e23a0',  # more-itertools: last-item/empty-iterable behavior
    'ce753d7eea707e6a',  # zobweyt/textcase: text boundary behavior
]
records = []
goldset_sha = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=GOLDSET_DIR, check=True, capture_output=True, text=True).stdout.strip()
print('Goldset revision:', goldset_sha)
for fingerprint in PILOT_FINGERPRINTS:
    if fingerprint not in records_by_fingerprint:
        raise RuntimeError(f'Pinned Goldset record not found: {fingerprint}')
    candidate = records_by_fingerprint[fingerprint]
    print(f"\nVerifying {fingerprint}: {candidate['repo']} ({candidate['license']})")
    verification = subprocess.run(
        [sys.executable, 'verify.py', '--corpus', 'oracles.jsonl', '--fingerprint', fingerprint],
        cwd=GOLDSET_DIR, capture_output=True, text=True,
    )
    print(verification.stdout)
    if verification.stderr.strip():
        print('Verifier messages:', verification.stderr[-2000:])
    if verification.returncode != 0 or not re.search(r'passed\s+1\s+failed\s+0\s+skipped\s+0', verification.stdout):
        raise RuntimeError(f"Goldset task {fingerprint} did not pass verification. Stop here and share this output so I can select an alternate.")
    records.append(candidate)
record = records[0]  # Backward-compatible convenience for inspecting the first task.
print('All three records passed their Goldset verifier. Continue to the model setup cells.')

Goldset runner module: /content/goldset/goldset/exec.py
Goldset revision: fe4f9bfbcb46637481242af2315c6b59951710b4

Verifying 1f881a956841fe4f: ilevkivskyi/com2ann (MIT)
verifying 1 record(s)

[1/1] pass  ilevkivskyi/com2ann  split_function_comment

passed 1   failed 0   skipped 0

Verifier messages: This clones public projects and runs their test suites on this machine. Use a container if that matters to you.



Verifying d2509b7ca26e23a0: more-itertools/more-itertools (MIT)
verifying 1 record(s)

[1/1] pass  more-itertools/more-itertools  last

passed 1   failed 0   skipped 0

Verifier messages: This clones public projects and runs their test suites on this machine. Use a container if that matters to you.



Verifying ce753d7eea707e6a: zobweyt/textcase (MIT)
verifying 1 record(s)

[1/1] pass  zobweyt/textcase  Boundary

passed 1   failed 0   skipped 0

Verifier messages: This clones public projects and runs their test suites on this machine. Use a container if that matters to you.





## Model choice for this runtime and install model libraries

The resource check selects a model tier using current free GPU memory and disk. The next code cell installs the required model libraries. Keep the chosen model unchanged across all six runs.

In [13]:
import subprocess
import sys

# Install only the model libraries. Keep Colab's existing PyTorch/CUDA build.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'transformers', 'accelerate', 'bitsandbytes'
], check=True)
from importlib.metadata import version
for package in ('transformers', 'accelerate', 'bitsandbytes'):
    print(f'{package}:', version(package))
print('Model libraries installed.')

transformers: 5.22.1
accelerate: 1.15.0
bitsandbytes: 0.50.2
Model libraries installed.


## Load the selected model and make one short request

This uses 4-bit NF4 compression. The model ID and pilot limits come from the fresh runtime resource report above. The same loaded model is reused for both conditions.

In [14]:
from pathlib import Path
import sys
import torch

PROJECT_DIR = Path('/content/mtech-bug-fix-agent')
SOURCE_DIR = PROJECT_DIR / 'src'
if not (SOURCE_DIR / 'bugfix_agent' / 'model.py').is_file():
    raise FileNotFoundError('Run the Download and install the project cell first.')
sys.path.insert(0, str(SOURCE_DIR))
from bugfix_agent.model import HuggingFaceModel

if 'MODEL_ID' not in globals():
    raise RuntimeError('Run the resource check cell first; it recommends a model using current free GPU memory and disk.')
if (
    'model' in globals()
    and isinstance(model, HuggingFaceModel)
    and model.model_id == MODEL_ID
    and getattr(model, '_model', None) is not None
):
    print('Reusing the model already loaded in this Colab runtime.')
else:
    import gc
    if 'model' in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()
    model = HuggingFaceModel(MODEL_ID, max_new_tokens=min(PILOT_MAX_NEW_TOKENS, 128), max_input_tokens=min(PILOT_MAX_INPUT_TOKENS, 1024))
torch.cuda.reset_peak_memory_stats()
print('Model loaded:', MODEL_ID)
print('Model snapshot:', model.revision)
print('Load time (seconds):', model.load_seconds)
print('GPU memory after load (GiB):', round(torch.cuda.memory_allocated() / (1024 ** 3), 2))

prompt = (
    'Return only a unified diff for this small Python bug. '
    'Current file: def divide(a, b): return a / b. '
    'Bug: when b is zero, return None instead of raising ZeroDivisionError.'
)
response = model.generate(prompt)
print('Generated response:')
print(response)
print('Token and time record:', model.last_usage)
print('Peak GPU memory (GiB):', round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2))
if not response.strip():
    raise RuntimeError('The model loaded but returned an empty response.')

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-Coder-3B-Instruct
Model snapshot: 488639f1ff808d1d3d0ba301aef8c11461451ec5
Load time (seconds): 326.203
GPU memory after load (GiB): 1.93
Generated response:
```diff
- def divide(a, b):
+ def divide(a, b):
     if b == 0:
-         return a / b
+         return None
```
Token and time record: {'input_tokens': 70, 'output_tokens': 38, 'generation_seconds': 3.382}
Peak GPU memory (GiB): 1.98



## Refinement round: run three Goldset tasks with and without retrieval

This development round uses the same three tasks and the already-selected model. The shared prompt now identifies whether the target is a class or function and tells the model to return only that replacement. Retrieval now excludes test and documentation folders and returns at most one chunk per file. This is a refinement check on these pilot tasks, not a final evaluation; keep these tasks out of the final held-out sample.

Run this only after **Download and install the project** has refreshed the code. It will make six attempts and save each result.

In [ ]:

import json
from pathlib import Path
import sys
import torch

PROJECT_DIR = Path('/content/mtech-bug-fix-agent')
SOURCE_DIR = PROJECT_DIR / 'src'
if not (SOURCE_DIR / 'bugfix_agent' / 'benchmark.py').is_file():
    raise FileNotFoundError('Run **Download and install the project** first so the updated project is available in Colab.')
if str(SOURCE_DIR) in sys.path:
    sys.path.remove(str(SOURCE_DIR))
sys.path.insert(0, str(SOURCE_DIR))
import bugfix_agent.benchmark as benchmark_module
run_goldset_record = benchmark_module.run_goldset_record
print('Using benchmark code from:', benchmark_module.__file__)

missing_steps = []
if 'RESOURCE_REPORT' not in globals() or 'MODEL_ID' not in globals():
    missing_steps.append('first resource-check code cell')
if 'PROJECT_DIR' not in globals():
    missing_steps.append('Download and install the project')
if 'records' not in globals() or len(globals().get('records', [])) != 3 or 'goldset_sha' not in globals():
    missing_steps.append('Verify the three Goldset pilot records')
if 'PILOT_MAX_INPUT_TOKENS' not in globals() or 'PILOT_MAX_NEW_TOKENS' not in globals():
    if 'first resource-check code cell' not in missing_steps:
        missing_steps.append('first resource-check code cell')
if 'model' not in globals():
    missing_steps.extend(['model-libraries installation code cell', 'Load the selected model and make one short request'])
if missing_steps:
    ordered = list(dict.fromkeys(missing_steps))
    raise RuntimeError('This Colab session is missing setup from: ' + ' → '.join(ordered) + '. Run each listed section, check its output, then retry this cell.')

model.max_new_tokens = PILOT_MAX_NEW_TOKENS
model.max_input_tokens = PILOT_MAX_INPUT_TOKENS
torch.cuda.empty_cache()
run_dir = PROJECT_DIR / 'artifacts' / 'runs' / 'goldset_pilot'
run_dir.mkdir(parents=True, exist_ok=True)
resource_path = run_dir / 'resource_report.json'
resource_path.write_text(json.dumps(RESOURCE_REPORT, indent=2), encoding='utf-8')
print('Saved resource report:', resource_path)
print('Same model for all tasks:', MODEL_ID, '| revision:', model.revision)
print('Token limits:', PILOT_MAX_INPUT_TOKENS, 'input /', PILOT_MAX_NEW_TOKENS, 'output')
pilot_records = []
for task_number, record in enumerate(records, start=1):
    print(f"\nTASK {task_number}/3: {record['fingerprint']} | {record['repo']} | {record['spec'][:180]}")
    for retrieval_enabled in (False, True):
        torch.cuda.empty_cache()
        result_path = run_goldset_record(
            record,
            model,
            run_dir,
            use_retrieval=retrieval_enabled,
            top_k=3,
            corpus_revision=goldset_sha,
        )
        result = json.loads(result_path.read_text(encoding='utf-8'))
        pilot_records.append(result)
        condition = 'Retrieval' if retrieval_enabled else 'No retrieval'
        print(condition + ':', result['test_status'])
        if result.get('skip_reason'):
            print('Reason:', result['skip_reason'])
        if result.get('failure_reason'):
            print('Reason:', result['failure_reason'])
        print('Buggy/reference checks:', result.get('buggy_revision_test_status'), '/', result.get('reference_fix_test_status'))
        print('Generation attempted:', 'model_response' in result)
        if result.get('failing_tests'):
            print('Failing tests:', result['failing_tests'])
        print('Run record:', result_path)
print('\nSummary by condition:')
for condition in ('no_retrieval', 'retrieval'):
    statuses = [r['test_status'] for r in pilot_records if r.get('condition') == condition]
    print(condition, ':', statuses.count('passed'), 'passed,', statuses.count('failed'), 'failed,', statuses.count('skipped'), 'skipped out of', len(statuses))
print('All task results were written to:', run_dir)
print('A failure is a result for that condition, not a reason to discard the other task results.')


## If a pilot task fails: inspect the saved run details

This reads the JSON records from the three-task pilot. It does not run the model again. If you have just restarted the runtime and it cannot find records, tell me; we will use the saved notebook output or rerun only the needed condition.

In [16]:

import json
from pathlib import Path

run_dir = Path('/content/mtech-bug-fix-agent/artifacts/runs/goldset_pilot')
selected_fingerprints = {item['fingerprint'] for item in records} if 'records' in globals() else set()
paths = sorted(run_dir.glob('*.json')) if run_dir.exists() else []
latest = {}
for path in paths:
    if path.name == 'resource_report.json':
        continue
    item = json.loads(path.read_text(encoding='utf-8'))
    fingerprint = item.get('benchmark_fingerprint')
    if fingerprint not in selected_fingerprints:
        continue
    key = (fingerprint, item.get('condition'))
    if key not in latest or item.get('created_at_utc', '') > latest[key][1].get('created_at_utc', ''):
        latest[key] = (path, item)
if not latest:
    print('No pilot run records found for the selected tasks in this runtime. Tell me and I will guide the recovery.')
else:
    for (fingerprint, condition), (path, item) in sorted(latest.items()):
        print('\n' + '=' * 72)
        print('Task:', fingerprint, '| repository:', item.get('repository'))
        print('Condition/result:', condition, '/', item.get('test_status'))
        print('Failure:', item.get('failure_reason', item.get('skip_reason', 'none')))
        print('Failing tests:', item.get('failing_tests', []))
        print('Buggy/reference checks:', item.get('buggy_revision_test_status'), '/', item.get('reference_fix_test_status'))
        print('Retrieved files:', [chunk.get('path') for chunk in item.get('retrieved_chunks', [])])
        if item.get('test_status') != 'passed':
            print('Generated replacement (first 3000 characters):\n', item.get('model_response', '<no model output>')[:3000])
        print('Run record:', path)



Task: 1f881a956841fe4f | repository: ilevkivskyi/com2ann
Condition/result: no_retrieval / failed
Failure: none
Buggy/reference checks: failed_as_expected / passed
Retrieved files: []
Generated replacement:
 from typing import List, Optional

def split_function_comment(
    comment: str, silent: bool = False
) -> Optional[List[str], str]:
    """Split function type comment into argument types and return types.

    This also removes any additional sub-comment. For example:

        # type: (int, str) -> None  # some explanation

    is transformed into: ['int', 'str'], 'None'.
    """
    typ, _ = split_sub_comment(comment)
    if "->" not in typ:
        if not silent:
            print("Invalid function type comment:", comment, file=sys.stderr)
        return None

    # TODO: ()->int vs () -> int -- keep spacing (also # type:int vs # type: int).
    arg_list, ret = typ.split("->")

    arg_list = arg_list.strip()
    ret = ret.strip()

    if not (arg_list[0] == "(" and arg_list[-1]